In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!jupyter nbconvert "/content/drive/My Drive/Colab Notebooks/Pattern Recognition Project/multi file solution/dataset.ipynb" --to python
!jupyter nbconvert "/content/drive/My Drive/Colab Notebooks/Pattern Recognition Project/multi file solution/modules.ipynb" --to python

In [2]:
import sys, os

py_file_location = "/content/drive/My Drive/Colab Notebooks/Pattern Recognition Project/multi file solution/"
sys.path.append(os.path.abspath(py_file_location))

import importlib, dataset, modules
importlib.reload(dataset)
importlib.reload(modules)

from dataset import *
from modules import *

/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


In [3]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

In [4]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Using:", device)

Using: cuda


In [5]:
train_dir = "/content/drive/MyDrive/Colab Notebooks/Pattern Recognition Project/AD_NC/train"
test_dir  = "/content/drive/MyDrive/Colab Notebooks/Pattern Recognition Project/AD_NC/test"

In [6]:
_, _, test_ds = create_datasets(train_dir, test_dir)
_, _, test_loader = create_dataloaders(_, _, test_ds)

In [7]:
model = ConvNeXt(in_chans=1, num_classes=2).to(device)
model.load_state_dict(torch.load("best_convnext_adni.pth", map_location=device))
model.eval()

FileNotFoundError: [Errno 2] No such file or directory: 'best_convnext_adni.pth'

In [8]:
import os
print(os.getcwd())
print(os.listdir())

/content
['.config', 'drive', 'sample_data']


In [ ]:
all_preds, all_targets = [], []
with torch.no_grad():
    for data, targets in test_loader:
        data, targets = data.to(device), targets.to(device)
        outputs = model(data)
        probs = F.softmax(outputs, dim=1)
        preds = probs.argmax(1)
        all_preds.extend(preds.cpu().numpy())
        all_targets.extend(targets.cpu().numpy())

In [ ]:
print("\n📋 Classification Report:")
print(classification_report(all_targets, all_preds, digits=4))

In [ ]:
cm = confusion_matrix(all_targets, all_preds)
disp = ConfusionMatrixDisplay(cm, display_labels=["AD", "NC"])
disp.plot(cmap='Reds'); plt.show()